# Ropedia Academy — D9 · Capstone: pixels to world memory

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChaoYue0307/ropedia-academy/blob/main/notebooks/D9.ipynb)

> **Runs the whole curriculum as one pipeline — poses → TSDF fusion → Bayesian semantics → scene-graph query — and names the cross-track reuse that makes it one system.**
>
> 把整门课程作为一条流水线运行——位姿 → TSDF 融合 → 贝叶斯语义 → 场景图查询——并点出让它成为一个系统的跨赛道复用。

This is the lesson's core example — **self-contained and runnable end to end**. It builds toy tensors, performs the lesson's key computation, and prints a real result, so you learn the concept by executing it.

Colab's default runtime already includes `torch`, `numpy`, and `networkx`, so just press **Run all** — every cell should go green. Sizes are shrunk to run on CPU; swap in a real batch and the same code scales up.

🔗 Full lesson (with the interactive demo & key terms): https://chaoyue0307.github.io/ropedia-academy/lesson/D9

In [ ]:
import numpy as np, networkx as nx
np.random.seed(0)

# CAPSTONE: pixels -> world memory. Wire every track into one runnable pipeline.
frames = [{"depth": 1.0 + np.random.randn()*0.05, "label": 2} for _ in range(20)]

# 1) POSES via SLAM/SfM (Track B bundle adjustment, online) — stubbed as identity.
poses = [np.eye(4) for _ in frames]

# 2) GEOMETRY: fuse depth into a TSDF (Track D3 = Track B's SDF, truncated).
tsdf, w = 0.0, 0
for f in frames:
    tsdf = (w*tsdf + np.clip(f["depth"] - 1.0, -1, 1)) / (w + 1); w += 1
print("fused surface offset (≈0):", round(tsdf, 3))

# 3) SEMANTICS: Bayes-fuse per-frame labels into the map (Track D4).
obj_class = int(np.bincount([f["label"] for f in frames]).argmax())
print("fused object class:", obj_class)

# 4) STRUCTURE: build + query a scene graph (Track D5).
g = nx.DiGraph(); g.add_edge(f"obj{obj_class}", "table", rel="on"); g.add_edge("table","kitchen", rel="in")
print("query 'is the object in the kitchen?':", nx.has_path(g, f"obj{obj_class}", "kitchen"))
print("cross-track reuse: BA<->SLAM back-end, SDF<->TSDF, CLIP<->open-vocab semantics")

### Where to go next

- Swap the toy tensors for a real batch and watch the shapes flow through.
- Open the matching lesson for the math and an interactive figure: https://chaoyue0307.github.io/ropedia-academy/lesson/D9
- Browse every notebook: https://github.com/ChaoYue0307/ropedia-academy/tree/main/notebooks